# AdaDetect Benchmark Notebook (GPU + TRUE CNN)

## Overview
This notebook benchmarks AdaDetect with various classifiers including a GPU-accelerated CNN and classical ML models.

## 1. INSTALL (run once)
```bash
pip install medmnist torchvision
```

## 2. IMPORTS

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
import medmnist
from medmnist import INFO

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from procedure import AdaDetectERM

np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
print(f"Using device: {device}")

## 3. DATA LOADERS (KEEP IMAGE SHAPE)

In [ ]:
def load_mnist():
    dataset = datasets.MNIST(root="./data", train=True, download=True)

    X = dataset.data.numpy() / 255.0  # (N, 28, 28)
    X = np.expand_dims(X, 1)          # (N, 1, 28, 28)
    y = dataset.targets.numpy()

    return X, y


def load_medmnist(name):
    info = INFO[name]
    DataClass = getattr(medmnist, info['python_class'])

    dataset = DataClass(split='train', download=True)

    X = dataset.imgs / 255.0  # already (N, H, W) or (N, H, W, C)

    if len(X.shape) == 3:
        X = np.expand_dims(X, 1)
    elif X.shape[-1] == 3:
        X = np.transpose(X, (0, 3, 1, 2))

    y = dataset.labels

    if len(y.shape) > 1 and y.shape[1] > 1:
        y = (y.sum(axis=1) > 0).astype(int)
    else:
        y = y.squeeze()

    return X, y


def get_datasets():
    return {
        "mnist": load_mnist(),
        "derma_mnist": load_medmnist("dermamnist"),
        "chest_mnist": load_medmnist("chestmnist"),
    }

print("Data loaders defined")

## 4. SPLIT

In [ ]:
def split_null_signal(X, y, null_class=0):
    mask_null = (y == null_class)
    return X, X[mask_null], ~mask_null

## 5. TRUE CNN CLASSIFIER (GPU)

In [ ]:
class CNNClassifier:
    def __init__(self, epochs=5, lr=1e-3, batch_size=128):
        self.epochs = epochs
        self.lr = lr
        self.batch_size = batch_size

        self.model = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        ).to(device)

    def fit(self, X, y):
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.long)

        loader = torch.utils.data.DataLoader(
            torch.utils.data.TensorDataset(X, y),
            batch_size=self.batch_size,
            shuffle=True,
            pin_memory=True
        )

        opt = optim.Adam(self.model.parameters(), lr=self.lr)
        loss_fn = nn.CrossEntropyLoss()

        self.model.train()
        for _ in range(self.epochs):
            for xb, yb in loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                opt.zero_grad()
                loss = loss_fn(self.model(xb), yb)
                loss.backward()
                opt.step()

    def predict_proba(self, X):
        X = torch.tensor(X, dtype=torch.float32)

        loader = torch.utils.data.DataLoader(X, batch_size=self.batch_size, pin_memory=True)

        self.model.eval()
        out = []

        with torch.no_grad():
            for xb in loader:
                xb = xb.to(device, non_blocking=True)
                probs = torch.softmax(self.model(xb), dim=1)
                out.append(probs.cpu())

        return torch.cat(out).numpy()

print("CNNClassifier defined")

## 6. TABULAR CLASSIFIERS (FLATTENED)

In [ ]:
def flatten(X):
    return X.reshape(len(X), -1)


def get_classifiers():
    return {
        "knn": KNeighborsClassifier(3),
        "svm": SVC(probability=True),
        "decision_tree": DecisionTreeClassifier(max_depth=5),
        "random_forest": RandomForestClassifier(n_estimators=100),
        "mlp": MLPClassifier(max_iter=300),
        "adaboost": AdaBoostClassifier(),
        "naive_bayes": GaussianNB(),
        "gaussian_process": GaussianProcessClassifier(1.0 * RBF(1.0)),
    }

print("Classifiers defined")

## 7. METRICS

In [ ]:
def compute_fdr_power(rejection_set, is_signal):
    R = len(rejection_set)
    if R == 0:
        return 0.0, 0.0

    rejected = np.zeros_like(is_signal, dtype=bool)
    rejected[rejection_set] = True

    V = np.sum(rejected & ~is_signal)
    S = np.sum(rejected & is_signal)

    return V / R, S / np.sum(is_signal)

## 8. RUN ADaDETECT

In [ ]:
def run_adadetect(model, x, xnull, is_signal, flatten_data=False):

    if flatten_data:
        x = flatten(x)
        xnull = flatten(xnull)

    proc = AdaDetectERM(scoring_fn=model, split_size=0.5)
    rejection_set = proc.apply(x=x, level=0.1, xnull=xnull)

    return compute_fdr_power(rejection_set, is_signal)

## 9. BENCHMARK

In [ ]:
def benchmark():
    datasets = get_datasets()
    results = []

    for dname, (X, y) in datasets.items():
        print(f"\nDataset: {dname}")

        x, xnull, is_signal = split_null_signal(X, y)

        # subsample
        idx = np.random.choice(len(x), 4000, replace=False)
        x, is_signal = x[idx], is_signal[idx]

        idx_null = np.random.choice(len(xnull), min(4000, len(xnull)), replace=False)
        xnull = xnull[idx_null]

        # ---- CNN (GPU) ----
        print("  Model: cnn")
        cnn = CNNClassifier()
        fdr, power = run_adadetect(cnn, x, xnull, is_signal, flatten_data=False)

        results.append({"dataset": dname, "model": "cnn", "fdr": fdr, "power": power})

        # ---- Classical models ----
        for name, model in get_classifiers().items():
            print(f"  Model: {name}")
            try:
                fdr, power = run_adadetect(model, x, xnull, is_signal, flatten_data=True)
                results.append({"dataset": dname, "model": name, "fdr": fdr, "power": power})
            except Exception as e:
                print("    failed:", e)

    return pd.DataFrame(results)

## 10. ANALYSIS

In [ ]:
def analyze(df):
    print(df.groupby(["dataset", "model"])[["fdr", "power"]].mean())

    for d in df.dataset.unique():
        sub = df[df.dataset == d]

        plt.figure()
        plt.scatter(sub.fdr, sub.power)

        for _, r in sub.iterrows():
            plt.text(r.fdr, r.power, r.model)

        plt.xlabel("FDR")
        plt.ylabel("Power")
        plt.title(d)
        plt.show()

## 11. INTERACTIVE EXECUTION CELLS
### Run ONE dataset at a time

In [ ]:
def prepare_dataset(name):
    datasets = get_datasets()
    X, y = datasets[name]

    x, xnull, is_signal = split_null_signal(X, y)

    # subsample for speed
    idx = np.random.choice(len(x), 4000, replace=False)
    x, is_signal = x[idx], is_signal[idx]

    idx_null = np.random.choice(len(xnull), min(4000, len(xnull)), replace=False)
    xnull = xnull[idx_null]

    return x, xnull, is_signal

### SELECT DATASET HERE
Uncomment one of the lines below to prepare a dataset:

In [ ]:
# x, xnull, is_signal = prepare_dataset("mnist")
# x, xnull, is_signal = prepare_dataset("derma_mnist")
# x, xnull, is_signal = prepare_dataset("chest_mnist")

## 12. RUN SINGLE MODEL CELLS

### CNN (GPU)

In [ ]:
# cnn = CNNClassifier()
# fdr, power = run_adadetect(cnn, x, xnull, is_signal, flatten_data=False)
# print("CNN -> FDR:", fdr, "Power:", power)

### SVM

In [ ]:
# svm = SVC(probability=True)
# fdr, power = run_adadetect(svm, x, xnull, is_signal, flatten_data=True)
# print("SVM -> FDR:", fdr, "Power:", power)

### Random Forest

In [ ]:
# rf = RandomForestClassifier(n_estimators=100)
# fdr, power = run_adadetect(rf, x, xnull, is_signal, flatten_data=True)
# print("RF -> FDR:", fdr, "Power:", power)

### KNN

In [ ]:
# knn = KNeighborsClassifier(3)
# fdr, power = run_adadetect(knn, x, xnull, is_signal, flatten_data=True)
# print("KNN -> FDR:", fdr, "Power:", power)

### MLP

In [ ]:
# mlp = MLPClassifier(max_iter=300)
# fdr, power = run_adadetect(mlp, x, xnull, is_signal, flatten_data=True)
# print("MLP -> FDR:", fdr, "Power:", power)

### AdaBoost

In [ ]:
# ada = AdaBoostClassifier()
# fdr, power = run_adadetect(ada, x, xnull, is_signal, flatten_data=True)
# print("AdaBoost -> FDR:", fdr, "Power:", power)

### Naive Bayes

In [ ]:
# nb = GaussianNB()
# fdr, power = run_adadetect(nb, x, xnull, is_signal, flatten_data=True)
# print("NB -> FDR:", fdr, "Power:", power)

### Gaussian Process (SLOW)

In [ ]:
# gp = GaussianProcessClassifier(1.0 * RBF(1.0))
# fdr, power = run_adadetect(gp, x, xnull, is_signal, flatten_data=True)
# print("GP -> FDR:", fdr, "Power:", power)

## 13. PARALLEL EXECUTION (MULTIPLE MODELS)

In [ ]:
from joblib import Parallel, delayed


def run_model(name, model, x, xnull, is_signal, flatten_flag):
    try:
        fdr, power = run_adadetect(model, x, xnull, is_signal, flatten_data=flatten_flag)
        return {"model": name, "fdr": fdr, "power": power}
    except Exception as e:
        return {"model": name, "fdr": None, "power": None, "error": str(e)}


def run_all_models_parallel(x, xnull, is_signal, n_jobs=2):
    tasks = [
        ("cnn", CNNClassifier(), False),
        ("svm", SVC(probability=True), True),
        ("rf", RandomForestClassifier(n_estimators=100), True),
        ("knn", KNeighborsClassifier(3), True),
        ("mlp", MLPClassifier(max_iter=300), True),
        ("adaboost", AdaBoostClassifier(), True),
        ("naive_bayes", GaussianNB(), True),
        ("gaussian_process", GaussianProcessClassifier(1.0 * RBF(1.0)), True),
    ]

    results = Parallel(n_jobs=n_jobs)(
        delayed(run_model)(name, model, x, xnull, is_signal, flatten_flag)
        for name, model, flatten_flag in tasks
    )

    return pd.DataFrame(results)

print("Parallel execution functions defined")

## 14. RUN PARALLEL
Uncomment to run parallel benchmarking:

In [ ]:
# Example:
# x, xnull, is_signal = prepare_dataset("mnist")
# df_parallel = run_all_models_parallel(x, xnull, is_signal, n_jobs=2)
# print(df_parallel)

## 15. NOTES ON PARALLELISM

- **n_jobs**: Controls how many models run simultaneously
- **GPU usage**: Keep `n_jobs=1` or `2` (CNN already uses GPU heavily)
- **CPU only**: You can increase `n_jobs` to number of cores
- **GaussianProcess**: Very slow → consider disabling it in parallel runs